In [1]:
import os
from typing import TypedDict,Literal
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool

from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph,END


C:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
API_KEY = 'API_KEY'

In [3]:
llm = ChatGoogleGenerativeAI(
    model = 'gemini-2.5-flash',
    google_api_key = API_KEY,
    temperature = 0.2
)

In [4]:
class AppointmentState(TypedDict):
    patient_id:str
    patient_name:str
    age: int
    fever_celsius:float
    oxygen_level:int
    heart_rate:int
    symptom_duration_days: int
    existing_conditions:list
 
    severity_level:str
    priority_level: str
    priority_reasoning: str
    consultation_type: str
    final_message: str

In [5]:
@tool
def assess_symptom_severity(
    fever_celsius: float,
    oxygen_level: int,
    heart_rate: int,
    symptom_duration_days: int,
) -> str:
    """
    Assesses patient symptom severity using clinical thresholds.
 
    Returns one of:
        CRITICAL  – requires immediate intervention
        MODERATE  – needs prompt but non-emergency care
        STABLE    – can be managed with routine consultation
    """
    critical_flags = 0
    moderate_flags = 0
 
    if fever_celsius >= 40.0:
        critical_flags += 1
    elif fever_celsius >= 38.5:
        moderate_flags += 1
 
    if oxygen_level < 90:
        critical_flags += 2
    elif oxygen_level < 95:
        moderate_flags += 1
 
    if heart_rate > 130 or heart_rate < 40:
        critical_flags += 1
    elif heart_rate > 100 or heart_rate < 50:
        moderate_flags += 1
 
    if symptom_duration_days >= 7:
        moderate_flags += 1
 
    if critical_flags >= 1:
        return "CRITICAL"
    if moderate_flags >= 2:
        return "MODERATE"
    return "STABLE"

In [6]:
def symptom_severity_node(state: AppointmentState) -> AppointmentState: 
    severity = assess_symptom_severity.invoke({
        "fever_celsius":state["fever_celsius"],
        "oxygen_level":state["oxygen_level"],
        "heart_rate": state["heart_rate"],
        "symptom_duration_days": state["symptom_duration_days"],
    })
 
    return {**state, "severity_level": severity}

In [7]:
def medical_prioritization_node(state: AppointmentState) -> AppointmentState:
    conditions_str = (
        ", ".join(state["existing_conditions"])
        if state["existing_conditions"]
        else "None"
    )
 
    system_prompt = (
        "You are an experienced hospital triage AI physician. "
        "Based on the patient's symptom severity and medical profile, "
        "classify the appointment priority. "
        "Reply with EXACTLY one of: EMERGENCY, PRIORITY_CONSULTATION, REGULAR_CONSULTATION "
        "on the first line, then a newline, then a one-sentence clinical justification."
    )
 
    user_prompt = (
        f"Patient Medical Profile:\n"
        f"- Name: {state['patient_name']}\n"
        f"- Age: {state['age']} years\n"
        f"- Assessed Severity: {state['severity_level']}\n"
        f"- Fever: {state['fever_celsius']} C\n"
        f"- Oxygen Level (SpO2): {state['oxygen_level']}%\n"
        f"- Heart Rate: {state['heart_rate']} BPM\n"
        f"- Symptom Duration: {state['symptom_duration_days']} days\n"
        f"- Existing Conditions: {conditions_str}\n\n"
        "Classify the appointment priority level."
    )
 
    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt),
    ])
 
    raw   = response.content.strip()
    lines = raw.split("\n", 1)
    priority_label = lines[0].strip()
    reasoning = lines[1].strip() if len(lines) > 1 else "No additional reasoning."
 
    for label in ["EMERGENCY", "PRIORITY_CONSULTATION", "REGULAR_CONSULTATION"]:
        if label in priority_label.upper():
            priority_label = label
            break
 
    print(f"Priority Level : {priority_label}")
    print(f"Reasoning : {reasoning}")
 
    return {**state, "priority_level": priority_label, "priority_reasoning": reasoning}

In [8]:
def consultation_assignment_node(state: AppointmentState) -> AppointmentState:
    assignment_map = {
        "EMERGENCY": (
            "ICU/ER",
            "Patient directed to ICU/Emergency Room for immediate intervention."
        ),
        "PRIORITY_CONSULTATION": (
            "Specialist Doctor",
            "Patient scheduled with a Specialist Doctor on priority basis."
        ),
        "REGULAR_CONSULTATION": (
            "General Physician",
            "Patient assigned to General Physician for routine consultation."
        ),
    }
 
    consultation_type, message = assignment_map.get(
        state["priority_level"],
        ("General Physician", "Defaulting to General Physician — priority unclear.")
    )
 
    print(f"Consultation Type : {consultation_type}")
    print(f"Message : {message}")
 
    return {**state, "consultation_type": consultation_type, "final_message": message}

In [9]:
def _build_appointment_graph():
    graph = StateGraph(AppointmentState)
 
    graph.add_node("symptom_severity", symptom_severity_node)
    graph.add_node("medical_prioritization", medical_prioritization_node)
    graph.add_node("consultation_assignment", consultation_assignment_node)
 
    graph.set_entry_point("symptom_severity")
    graph.add_edge("symptom_severity", "medical_prioritization")
    graph.add_edge("medical_prioritization", "consultation_assignment")
    graph.add_edge("consultation_assignment", END)
 
    compiled = graph.compile()
    return compiled
 
appointment_workflow = _build_appointment_graph()

In [10]:
def run_appointment_workflow(patient: dict) -> AppointmentState:
    result = appointment_workflow.invoke(patient)
    print(f"\n FINAL SUMMARY")
    print(f"Patient : {result['patient_name']} (Age {result['age']})")
    print(f"Severity Level : {result['severity_level']}")
    print(f"Priority Level : {result['priority_level']}")
    print(f"Consultation Type : {result['consultation_type']}")
    print(f"Message : {result['final_message']}")
    return result

In [11]:
if __name__ == "__main__":
 
    test_patients = [
        # --- Patient A: Critical (low SpO2, high fever) ---
        {
            "patient_id":             "PAT001",
            "patient_name":           "Sunita Rao",
            "age":                    72,
            "fever_celsius":          40.2,
            "oxygen_level":           86,
            "heart_rate":             135,
            "symptom_duration_days":  3,
            "existing_conditions":    ["diabetes", "hypertension"],
            "severity_level":         "",
            "priority_level":         "",
            "priority_reasoning":     "",
            "consultation_type":      "",
            "final_message":          "",
        },
        # --- Patient B: Moderate ---
        {
            "patient_id":             "PAT002",
            "patient_name":           "Arjun Mehta",
            "age":                    45,
            "fever_celsius":          38.9,
            "oxygen_level":           93,
            "heart_rate":             105,
            "symptom_duration_days":  5,
            "existing_conditions":    ["asthma"],
            "severity_level":         "",
            "priority_level":         "",
            "priority_reasoning":     "",
            "consultation_type":      "",
            "final_message":          "",
        },
        # --- Patient C: Stable / Routine ---
        {
            "patient_id":             "PAT003",
            "patient_name":           "Kavya Nair",
            "age":                    28,
            "fever_celsius":          37.2,
            "oxygen_level":           98,
            "heart_rate":             78,
            "symptom_duration_days":  2,
            "existing_conditions":    [],
            "severity_level":         "",
            "priority_level":         "",
            "priority_reasoning":     "",
            "consultation_type":      "",
            "final_message":          "",
        },
    ]
 
    for patient in test_patients:
        run_appointment_workflow(patient)
        print()

Priority Level : EMERGENCY
Reasoning : The patient presents with critical severity, severe hypoxemia (SpO2 86%), high fever (40.2 C), tachycardia (135 BPM), and significant comorbidities in an elderly individual, indicating an immediate life-threatening condition.
Consultation Type : ICU/ER
Message : Patient directed to ICU/Emergency Room for immediate intervention.

 FINAL SUMMARY
Patient : Sunita Rao (Age 72)
Severity Level : CRITICAL
Priority Level : EMERGENCY
Consultation Type : ICU/ER
Message : Patient directed to ICU/Emergency Room for immediate intervention.

Priority Level : PRIORITY_CONSULTATION
Reasoning : The patient requires urgent assessment due to concerning oxygen saturation (93%) and tachycardia in the context of asthma and persistent fever for five days, indicating potential respiratory compromise.
Consultation Type : Specialist Doctor
Message : Patient scheduled with a Specialist Doctor on priority basis.

 FINAL SUMMARY
Patient : Arjun Mehta (Age 45)
Severity Level :